# LeetCode 353 — Design Snake Game bằng Python

Link đề: https://leetcode.com/problems/design-snake-game/ · Độ khó: **Medium** · Chủ đề: Design (thiết kế lớp), Queue (hàng đợi), Hash Table (bảng băm), Simulation (mô phỏng), Array

Skeleton LeetCode:

```python
class SnakeGame:

    def __init__(self, width: int, height: int, food: List[List[int]]):
        ...

    def move(self, direction: str) -> int:
        ...


# Your SnakeGame object will be instantiated and called as such:
# obj = SnakeGame(width, height, food)
# param_1 = obj.move(direction)
```

Ghi chú: đây là bài **Premium**; đề được viết theo hiểu biết về bài (ví dụ 1 lấy theo mô tả phổ biến của bài), nên **đối chiếu lại với đề gốc nếu bạn có tài khoản Premium**. Mọi đáp án trong notebook đã được kiểm chứng bằng code (so với một bộ mô phỏng độc lập). Các ví dụ tôi tự dựng thêm (như bàn `2 × 2` để đuôi rời đi) là **tự dựng**, được kiểm chứng bằng oracle chứ không lấy từ đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **thiết kế lớp** `SnakeGame`: bàn chơi `height × width`, rắn bắt đầu ở `(0, 0)` dài `1`, mỗi lần `move(direction)` rắn tiến một ô; trả về **điểm** (số mồi đã ăn) hoặc `-1` nếu **thua** (đâm tường hoặc đâm vào thân).
2. Nắm các công cụ Python: **tuple vs list** (`(1, 2) == [1, 2]` là `False`), `set`, `collections.deque` (`append`, `popleft`), từ điển hướng đi `{"U": (-1, 0), ...}`, so sánh dây chuyền `0 <= r < height`, và `dict` ghi thời điểm.
3. Đi từ **thân rắn là `list` với `pop(0)` và `in` (`O(L)`)** → **`deque` + `set` (`O(1)`)** → **không cần hàng đợi: ghi "thời điểm đầu rắn đến" mỗi ô trong `dict`** (`O(1)`).
4. Hiểu mấu chốt: thân rắn là một **hàng đợi vào-trước-ra-trước**: đầu thêm vào cuối, đuôi lấy ra ở đầu; **đuôi rời đi trước khi** ta kiểm tra va chạm (trừ khi vừa ăn mồi thì đuôi **ở lại**).
5. Tự dựng **bộ chạy chuỗi lệnh** và **oracle mô phỏng độc lập**, rồi kiểm thử bằng ván ngẫu nhiên có seed (đúng luật: mồi không xuất hiện trên thân rắn).

## 2. Đề bài & ví dụ

Thiết kế trò chơi **rắn săn mồi** chạy trên màn hình kích thước `height × width`.

- Rắn bắt đầu ở góc trên-trái `(0, 0)` với độ dài `1`.
- Cho mảng `food` với `food[i] = (rᵢ, cᵢ)` là **hàng và cột** của một mồi. Khi rắn ăn mồi, **độ dài và điểm** cùng tăng `1`.
- Các mồi xuất hiện **lần lượt**: mồi thứ `i + 1` chỉ xuất hiện khi rắn đã ăn mồi thứ `i`. Khi một mồi xuất hiện, **đảm bảo** nó **không** nằm trên ô đang có thân rắn.
- Trò chơi **kết thúc** nếu rắn **ra ngoài biên** hoặc **đầu chạm vào thân** (sau khi rắn đã di chuyển; ví dụ rắn dài `4` không thể tự đâm vào chính nó).

Cài đặt lớp `SnakeGame`:

- `SnakeGame(int width, int height, int[][] food)`: khởi tạo màn hình `height × width` và danh sách mồi.
- `int move(String direction)`: rắn đi một bước theo `direction` (`'U'` lên, `'D'` xuống, `'L'` trái, `'R'` phải). Trả về **điểm** sau bước đi; nếu trò chơi kết thúc thì trả `-1`.

**Ví dụ 1** (màn `3` cột × `2` hàng, mồi tại `(1, 2)` rồi `(0, 1)`)

```text
Input
["SnakeGame", "move", "move", "move", "move", "move", "move"]
[[3, 2, [[1, 2], [0, 1]]], ["R"], ["D"], ["R"], ["U"], ["L"], ["U"]]
Output
[null, 0, 0, 1, 1, 2, -1]
```

Giải thích: `R` → `(0,1)`; `D` → `(1,1)`; `R` → `(1,2)` ăn mồi thứ nhất (điểm `1`), mồi thứ hai hiện tại `(0,1)`; `U` → `(0,2)`; `L` → `(0,1)` ăn mồi thứ hai (điểm `2`); `U` → ra ngoài biên trên ⇒ `-1`.

**Ràng buộc:**

- `1 <= width, height <= 10^4`
- `1 <= food.length <= 50`
- `food[i].length == 2`, `0 <= rᵢ < height`, `0 <= cᵢ < width`
- `direction` là `'U'`, `'D'`, `'L'` hoặc `'R'`
- Có tối đa `10^4` lần gọi `move`

In [1]:
ops = ["SnakeGame", "move", "move", "move", "move", "move", "move"]
args = [[3, 2, [[1, 2], [0, 1]]], ["R"], ["D"], ["R"], ["U"], ["L"], ["U"]]
expected = [None, 0, 0, 1, 1, 2, -1]

print(list(zip(ops, args)))
print("kỳ vọng:", expected)

[('SnakeGame', [3, 2, [[1, 2], [0, 1]]]), ('move', ['R']), ('move', ['D']), ('move', ['R']), ('move', ['U']), ('move', ['L']), ('move', ['U'])]
kỳ vọng: [None, 0, 0, 1, 1, 2, -1]


## 3. Kiến thức nền

### 3.1. Lớp, `self` và trạng thái giữ lại giữa các lần gọi

Ở bài thiết kế ta viết **một lớp**: `__init__` chạy **một lần** khi tạo đối tượng và gán các thuộc tính `self.xxx` (trạng thái); `move` dùng và **cập nhật** trạng thái đó nên kết quả lần sau phụ thuộc các lần trước. Ở đây trạng thái gồm: vị trí đầu, thân rắn, điểm, chỉ số mồi hiện tại. Mục tiêu thiết kế: chọn cách lưu để `move` chạy **nhanh**.

In [2]:
class Counter3:
    def __init__(self, start):
        self.value = start

    def add(self, delta):
        self.value += delta
        return self.value


c = Counter3(10)
print(c.add(1), c.add(5))                  # 11 16  (trạng thái được giữ giữa các lần gọi)

11 16


### 3.2. Cách LeetCode gọi lớp: bộ chạy chuỗi lệnh

Ví dụ gồm hai danh sách song song: `ops` (tên lệnh, phần tử đầu là tên lớp) và `args` (tham số). Bộ chạy `run_commands` tạo đối tượng ở lệnh đầu rồi gọi phương thức theo tên bằng `getattr(obj, tên)(*tham_số)`; hàm khởi tạo cho `None` (đề viết `null`).

In [3]:
def run_commands(cls, ops, args):
    """Mô phỏng cách LeetCode gọi: ops[0] là tên lớp (khởi tạo), các lệnh sau là tên phương thức."""
    obj = None
    out = []
    for name, a in zip(ops, args):
        if obj is None:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, name)(*a))
    return out


print(run_commands(Counter3, ["Counter3", "add", "add"], [[10], [1], [5]]))     # [None, 11, 16]

[None, 11, 16]


### 3.3. Tuple và list: vì sao `[1, 2] == (1, 2)` là `False`

Tọa độ ô thường lưu bằng **tuple** `(r, c)`: tuple là **bất biến** nên **băm được**, bỏ vào `set` hoặc làm khóa `dict` được; `list` thì **không** (báo `TypeError`). Cũng quan trọng: **`[1, 2] == (1, 2)` là `False`** (khác kiểu). `food[i]` của đề là **list** `[r, c]`, nên khi so với tọa độ rắn (tuple) ta phải đổi: `tuple(food[i])`.

In [4]:
print([1, 2] == (1, 2), tuple([1, 2]) == (1, 2))         # False True
cells = {(0, 0), (0, 1)}
print((0, 1) in cells, (5, 5) in cells)                  # True False
try:
    {[0, 0]}                                              # list không băm được
except TypeError as e:
    print("TypeError:", e)

False True
True False
TypeError: unhashable type: 'list'


### 3.4. `deque`: hàng đợi hai đầu

`list.pop(0)` lấy phần tử đầu nhưng phải **dịch cả danh sách** ⇒ `O(L)`. `collections.deque` là **hàng đợi hai đầu**: `append(x)` thêm vào cuối và `popleft()` lấy ở đầu đều **`O(1)`**. Thân rắn đúng là hàng đợi: **đầu mới thêm vào cuối**, **đuôi lấy ra ở đầu**. `set` cho `x in s`, `s.add(x)`, `s.remove(x)` đều `O(1)` trung bình.

In [5]:
from collections import deque

body = deque([(0, 0)])
body.append((0, 1))                    # đầu mới
body.append((1, 1))
print(body)                             # deque([(0, 0), (0, 1), (1, 1)])
print(body.popleft(), body)             # (0, 0) deque([(0, 1), (1, 1)])   <- đuôi rời đi
print(body[0], body[-1])                # (0, 1) (1, 1)   đuôi và đầu

deque([(0, 0), (0, 1), (1, 1)])
(0, 0) deque([(0, 1), (1, 1)])
(0, 1) (1, 1)


### 3.5. Hướng đi và quy ước (hàng, cột)

Tọa độ `(r, c)`: `r` là **hàng** (từ trên xuống, `0` ở trên cùng), `c` là **cột** (từ trái sang). Vì vậy `'U'` (lên) làm **`r` giảm**, `'D'` làm `r` tăng, `'L'` làm `c` giảm, `'R'` làm `c` tăng. Lưu bằng một `dict` hướng → `(dr, dc)`. Kiểm tra biên bằng so sánh dây chuyền: `0 <= r < height and 0 <= c < width`.

In [6]:
DIRS = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}
r, c = 0, 1
for d in "RDLU":
    dr, dc = DIRS[d]
    r, c = r + dr, c + dc
    print(d, "->", (r, c))
height, width = 2, 3
print(0 <= 1 < height, 0 <= 2 < width, 0 <= -1 < height)      # True True False

R -> (0, 2)
D -> (1, 2)
L -> (1, 1)
U -> (0, 1)
True True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Đi lại **Ví dụ 1** (`width = 3`, `height = 2`, `S` là đầu, `s` là thân, `F` là mồi hiện tại, `.` là trống). Mồi thứ nhất ở `(1,2)`.

| Bước | Di chuyển | Đầu mới | Bàn chơi sau bước | Điểm |
|---|---|---|---|---|
| 0 | khởi tạo | `(0,0)` | `S . .` / `. . F` | 0 |
| 1 | `R` | `(0,1)` | `. S .` / `. . F` (đuôi `(0,0)` rời đi) | 0 |
| 2 | `D` | `(1,1)` | `. . .` / `. S F` | 0 |
| 3 | `R` | `(1,2)` | ăn mồi ⇒ rắn dài 2: `. . .` / `. s S` (mồi mới ở `(0,1)`) | 1 |
| 4 | `U` | `(0,2)` | `. F S` / `. . s`  (đuôi `(1,1)` rời đi) | 1 |
| 5 | `L` | `(0,1)` | ăn mồi ⇒ dài 3: `. S s` / `. . s` | 2 |
| 6 | `U` | `(-1,1)` | **ra ngoài biên trên** | **−1** |

Ba nhận xét làm nên lời giải:

1. Mỗi lần đi **không ăn mồi**, **đầu thêm một ô** và **đuôi bỏ một ô** (độ dài giữ nguyên). Khi **ăn mồi**, đuôi **ở lại** (rắn dài thêm 1).
2. Thân rắn là hàng đợi: ô đưa vào **lâu nhất** là ô đầu tiên được rút ra.
3. Va chạm với thân chỉ xét **sau khi** đuôi đã rời đi, vì ô vừa rời là ô trống ngay.

In [7]:
def render(width, height, body, head, food_cell=None):
    grid = [["." for _ in range(width)] for _ in range(height)]
    for (r, c) in body:
        grid[r][c] = "s"
    if food_cell is not None:
        grid[food_cell[0]][food_cell[1]] = "F"
    grid[head[0]][head[1]] = "S"
    return "\n".join(" ".join(row) for row in grid)


print(render(3, 2, [(1, 1), (1, 2)], (1, 2), (0, 1)))         # trạng thái sau bước 3

. F .
. s S


## 5. Cách 1 — Brute Force / baseline: thân rắn là `list`

**Ý tưởng.** Lưu thân rắn bằng **một `list` các tuple**, đuôi ở **đầu** list, đầu ở **cuối**. Mỗi `move`:

1. Tính đầu mới `(nr, nc)`; ra biên ⇒ `-1`.
2. Nếu đầu mới trùng mồi hiện tại: **ăn** (điểm tăng, **không** bỏ đuôi); ngược lại **bỏ đuôi** bằng `pop(0)`.
3. Nếu đầu mới **thuộc thân** (kiểm tra `in` trên list) ⇒ `-1`.
4. Thêm đầu mới vào cuối list.

In [8]:
from typing import List

DIRS = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}


class SnakeGameList:
    def __init__(self, width: int, height: int, food: List[List[int]]):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]       # đổi list [r, c] thành tuple (r, c)
        self.fi = 0                                # chỉ số mồi hiện tại
        self.score = 0
        self.body = [(0, 0)]                       # body[0] là đuôi, body[-1] là đầu

    def move(self, direction: str) -> int:
        dr, dc = DIRS[direction]
        nr, nc = self.body[-1][0] + dr, self.body[-1][1] + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1                              # đâm tường
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1                           # ăn mồi: đuôi ở lại
            self.score += 1
        else:
            self.body.pop(0)                       # đuôi rời đi: O(L)
        if (nr, nc) in self.body:                  # kiểm tra tự đâm: O(L)
            return -1
        self.body.append((nr, nc))
        return self.score


print(run_commands(SnakeGameList, ops, args), "(kỳ vọng", expected, ")")

[None, 0, 0, 1, 1, 2, -1] (kỳ vọng [None, 0, 0, 1, 1, 2, -1] )


**Độ phức tạp (Cách 1).** Gọi `L` là độ dài rắn. Mỗi `move`: `pop(0)` tốn `O(L)` và `in` trên `list` tốn `O(L)` ⇒ **Time `O(L)` mỗi bước**; Space `O(L)`.

**Chú ý:** với ràng buộc của đề (`food.length <= 50` ⇒ `L <= 51`) cách này **vẫn đủ nhanh** (mỗi bước cỡ vài chục phép). Nó chỉ chậm khi `L` lớn (ví dụ nhiều nghìn mồi), và cũng là câu hỏi phỏng vấn kinh điển: "làm sao để mỗi bước `O(1)`?".

**Nút thắt:** hai thao tác `O(L)`: **lấy đuôi ra khỏi đầu list** và **kiểm tra ô có thuộc thân không**. Ta cần cấu trúc cho mỗi thao tác `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Thân rắn = **hàng đợi** (vào-trước-ra-trước) **và** là một **tập ô bị chiếm**. Dùng hai cấu trúc **đồng bộ**:

- `deque` `body`: đầu mới `append` vào cuối, đuôi `popleft` ở đầu: cả hai `O(1)`.
- `set` `occupied`: chứa đúng các ô trong `body`; kiểm tra thuộc thân `O(1)`; khi đuôi rời đi thì **`remove` khỏi set**; khi đầu mới vào thì **`add` vào set**.

**Thứ tự quan trọng:** (1) tính đầu mới; (2) kiểm tra biên; (3) **nếu không ăn thì `popleft` đuôi và `remove` khỏi set**; (4) **sau đó** kiểm tra đầu mới có trong `occupied` không; (5) thêm đầu mới. Bước (3) **trước** bước (4) cho phép đầu đi vào ô mà đuôi **vừa** rời.

Một góc nhìn khác (Cách 3): thân rắn chính là **`L` vị trí gần nhất của đầu** theo thời gian. Nếu ghi lại "**thời điểm đầu rắn lần cuối ở ô nào**" thì biết ngay ô nào còn bị thân chiếm, **không cần hàng đợi**.

In [9]:
# Minh họa quy tắc "đuôi rời đi trước khi kiểm tra va chạm": rắn dài 4 trong bàn 2x2
# thân (đuôi -> đầu): (0,0) (0,1) (1,1) (1,0); đi 'U' tới (0,0) chính là ô của ĐUÔI
body = deque([(0, 0), (0, 1), (1, 1), (1, 0)])
occupied = set(body)
head = body[-1]
new_head = (head[0] - 1, head[1])
print("đầu mới:", new_head, "| ô này đang bị chiếm (trước khi bỏ đuôi):", new_head in occupied)
tail = body.popleft()
occupied.remove(tail)
print("sau khi đuôi", tail, "rời đi: ô bị chiếm =", new_head in occupied)    # False -> được phép đi vào

đầu mới: (0, 0) | ô này đang bị chiếm (trước khi bỏ đuôi): True
sau khi đuôi (0, 0) rời đi: ô bị chiếm = False


## 7. Các cách tiếp theo

### Cách 2 — `deque` + `set` (mỗi `move` đều `O(1)`)

**Giải thích từng đoạn.**

1. `self.body = deque([(0, 0)])` và `self.occupied = {(0, 0)}`: đầu là `body[-1]`.
2. Tính `(nr, nc)`; ngoài biên ⇒ `-1`.
3. Đầu mới trùng `food[fi]` ⇒ ăn: `fi += 1`, `score += 1`, **không** bỏ đuôi. Ngược lại: `tail = body.popleft()`, `occupied.remove(tail)`.
4. `(nr, nc) in occupied` ⇒ `-1`.
5. `body.append((nr, nc))`, `occupied.add((nr, nc))`, trả `score`.

In [10]:
from collections import deque


class SnakeGameDequeDemo:
    def __init__(self, width: int, height: int, food: List[List[int]]):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.score = 0
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction: str) -> int:
        dr, dc = DIRS[direction]
        r, c = self.body[-1]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1
            self.score += 1                          # ăn mồi: đuôi ở lại
        else:
            tail = self.body.popleft()               # đuôi rời đi trước khi kiểm tra va chạm
            self.occupied.remove(tail)
        if (nr, nc) in self.occupied:
            return -1
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return self.score


print(run_commands(SnakeGameDequeDemo, ops, args), "(kỳ vọng", expected, ")")

[None, 0, 0, 1, 1, 2, -1] (kỳ vọng [None, 0, 0, 1, 1, 2, -1] )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành SnakeGame khi nộp)
from collections import deque
from typing import List

class SnakeGameDeque:
    def __init__(self, width: int, height: int, food: List[List[int]]):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction: str) -> int:
        dr, dc = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}[direction]
        r, c = self.body[-1]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1
        else:
            self.occupied.remove(self.body.popleft())
        if (nr, nc) in self.occupied:
            return -1
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return len(self.body) - 1                    # điểm = độ dài - 1


print(run_commands(SnakeGameDeque, ops, args))

[None, 0, 0, 1, 1, 2, -1]


**Độ phức tạp Cách 2:** mỗi `move`: `popleft`, `remove`, `in`, `append`, `add` đều `O(1)` (trung bình với `set`) ⇒ **Time `O(1)` mỗi bước**, **Space `O(L)`** cho `deque` và `set`.

Nhận xét nhỏ: **điểm = độ dài − 1** (mỗi mồi làm dài thêm 1 từ độ dài ban đầu 1), nên có thể bỏ biến `score` riêng (bản nộp ở trên dùng `len(self.body) - 1`).

### Cách 3 — Ghi "thời điểm đầu rắn đến" mỗi ô (không cần hàng đợi)

**Ý tưởng.** Thân rắn dài `L` gồm đúng **`L` ô mà đầu rắn đã đi qua gần nhất**. Gọi `t` là **số bước đã đi** (đầu ở ô gốc tại `t = 0`). Lưu `last[ô] = t` là **lần cuối đầu rắn đứng ở ô đó**. Trước bước mới (sẽ là bước `t`), thân gồm các ô có `last` thuộc `[t − L, t − 1]`.

- **Không ăn mồi:** đuôi (đi tại thời điểm `t − L`) rời đi, thân còn `[t − L + 1, t − 1]`. Đầu mới va chạm **khi và chỉ khi** nó thuộc `last` và `t − last[ô] < L`, tức `last[ô] > t − L`.
- **Ăn mồi:** đuôi ở lại; mồi không xuất hiện trên thân nên không cần kiểm tra va chạm; `L += 1`.
- Cập nhật `last[ô mới] = t`.

Không dùng `deque`; điểm là `L − 1`. Mỗi bước đều `O(1)`.

In [12]:
class SnakeGameTimeDemo:
    def __init__(self, width: int, height: int, food: List[List[int]]):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.length = 1                              # độ dài rắn L
        self.t = 0                                   # số bước đã đi
        self.head = (0, 0)
        self.last = {(0, 0): 0}                      # ô -> lần cuối đầu rắn đứng ở ô đó

    def move(self, direction: str) -> int:
        dr, dc = DIRS[direction]
        nr, nc = self.head[0] + dr, self.head[1] + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1
        self.t += 1
        cell = (nr, nc)
        if self.fi < len(self.food) and self.food[self.fi] == cell:
            self.fi += 1
            self.length += 1                         # ăn mồi: đuôi ở lại
        elif cell in self.last and self.t - self.last[cell] < self.length:
            return -1                                # ô này vẫn còn thân (sau khi đuôi rời đi)
        self.last[cell] = self.t
        self.head = cell
        return self.length - 1


print(run_commands(SnakeGameTimeDemo, ops, args), "(kỳ vọng", expected, ")")

[None, 0, 0, 1, 1, 2, -1] (kỳ vọng [None, 0, 0, 1, 1, 2, -1] )


In [13]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành SnakeGame khi nộp)
from typing import List

class SnakeGameTime:
    def __init__(self, width: int, height: int, food: List[List[int]]):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.length = 1
        self.t = 0
        self.head = (0, 0)
        self.last = {(0, 0): 0}

    def move(self, direction: str) -> int:
        dr, dc = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}[direction]
        nr, nc = self.head[0] + dr, self.head[1] + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1
        self.t += 1
        cell = (nr, nc)
        if self.fi < len(self.food) and self.food[self.fi] == cell:
            self.fi += 1
            self.length += 1
        elif cell in self.last and self.t - self.last[cell] < self.length:
            return -1
        self.last[cell] = self.t
        self.head = cell
        return self.length - 1


print(run_commands(SnakeGameTime, ops, args))

[None, 0, 0, 1, 1, 2, -1]


**Độ phức tạp Cách 3:** mỗi `move` là vài phép trên `dict` ⇒ **Time `O(1)` mỗi bước**; **Space `O(K)`** với `K` là số ô **khác nhau** mà đầu từng đi qua (`<=` số bước đã gọi, tối đa `10⁴`), không phụ thuộc `L`. So với Cách 2, bộ nhớ có thể lớn hơn một chút (nhớ cả ô cũ), nhưng code **không** cần đồng bộ hai cấu trúc.

## 8. TRACE từng bước

Ta trace Cách 2 (`deque` + `set`) trên **Ví dụ 1**, rồi Cách 3 (thời điểm) trên ván `2 × 2` cho thấy **đuôi rời đi** (rắn dài `4` mà vẫn đi vào ô đuôi cũ). Ván `2 × 2` là **ví dụ tự dựng**: `food = [[0,1],[1,1],[1,0]]`, các bước `R, D, L, U, R`.

In [14]:
def trace_deque(width, height, food, directions):
    food = [tuple(f) for f in food]
    body, occupied, fi = deque([(0, 0)]), {(0, 0)}, 0
    print(f"màn {height}x{width}, mồi = {food}")
    results = []
    for step, d in enumerate(directions, 1):
        dr, dc = DIRS[d]
        r, c = body[-1]
        nr, nc = r + dr, c + dc
        print(f"bước {step}: '{d}' đầu {(r, c)} -> {(nr, nc)}")
        if not (0 <= nr < height and 0 <= nc < width):
            print("   ngoài biên => -1")
            results.append(-1)
            break
        if fi < len(food) and food[fi] == (nr, nc):
            fi += 1
            print(f"   ăn mồi thứ {fi}, đuôi ở lại")
        else:
            tail = body.popleft()
            occupied.remove(tail)
            print(f"   không ăn: đuôi {tail} rời đi")
        if (nr, nc) in occupied:
            print("   đâm vào thân => -1")
            results.append(-1)
            break
        body.append((nr, nc))
        occupied.add((nr, nc))
        print(f"   body = {list(body)}, occupied = {sorted(occupied)}, điểm = {len(body) - 1}")
        results.append(len(body) - 1)
    return results


assert trace_deque(3, 2, [[1, 2], [0, 1]], "RDRULU") == [0, 0, 1, 1, 2, -1]

màn 2x3, mồi = [(1, 2), (0, 1)]
bước 1: 'R' đầu (0, 0) -> (0, 1)
   không ăn: đuôi (0, 0) rời đi
   body = [(0, 1)], occupied = [(0, 1)], điểm = 0
bước 2: 'D' đầu (0, 1) -> (1, 1)
   không ăn: đuôi (0, 1) rời đi
   body = [(1, 1)], occupied = [(1, 1)], điểm = 0
bước 3: 'R' đầu (1, 1) -> (1, 2)
   ăn mồi thứ 1, đuôi ở lại
   body = [(1, 1), (1, 2)], occupied = [(1, 1), (1, 2)], điểm = 1
bước 4: 'U' đầu (1, 2) -> (0, 2)
   không ăn: đuôi (1, 1) rời đi
   body = [(1, 2), (0, 2)], occupied = [(0, 2), (1, 2)], điểm = 1
bước 5: 'L' đầu (0, 2) -> (0, 1)
   ăn mồi thứ 2, đuôi ở lại
   body = [(1, 2), (0, 2), (0, 1)], occupied = [(0, 1), (0, 2), (1, 2)], điểm = 2
bước 6: 'U' đầu (0, 1) -> (-1, 1)
   ngoài biên => -1


In [15]:
def trace_time(width, height, food, directions):
    food = [tuple(f) for f in food]
    length, t, head, fi = 1, 0, (0, 0), 0
    last = {(0, 0): 0}
    print(f"màn {height}x{width}, mồi = {food}")
    results = []
    for d in directions:
        dr, dc = DIRS[d]
        cell = (head[0] + dr, head[1] + dc)
        if not (0 <= cell[0] < height and 0 <= cell[1] < width):
            print(f"'{d}' -> {cell}: ngoài biên => -1")
            results.append(-1)
            break
        t += 1
        if fi < len(food) and food[fi] == cell:
            fi += 1
            length += 1
            print(f"t={t} '{d}' -> {cell}: ăn mồi, L={length}")
        elif cell in last and t - last[cell] < length:
            print(f"t={t} '{d}' -> {cell}: last={last[cell]}, t - last = {t - last[cell]} < L={length} => va chạm, -1")
            results.append(-1)
            break
        else:
            seen = f"last={last[cell]}, t - last = {t - last[cell]} >= L={length}: đuôi đã rời, được phép" if cell in last else "ô mới"
            print(f"t={t} '{d}' -> {cell}: {seen}")
        last[cell] = t
        head = cell
        results.append(length - 1)
    return results


assert trace_time(2, 2, [[0, 1], [1, 1], [1, 0]], "RDLUR") == [1, 2, 3, 3, 3]

màn 2x2, mồi = [(0, 1), (1, 1), (1, 0)]
t=1 'R' -> (0, 1): ăn mồi, L=2
t=2 'D' -> (1, 1): ăn mồi, L=3
t=3 'L' -> (1, 0): ăn mồi, L=4
t=4 'U' -> (0, 0): last=0, t - last = 4 >= L=4: đuôi đã rời, được phép
t=5 'R' -> (0, 1): last=1, t - last = 4 >= L=4: đuôi đã rời, được phép


Bảng trace Cách 2 với Ví dụ 1:

| bước | hướng | đầu mới | hành động | `body` (đuôi → đầu) | điểm |
|---|---|---|---|---|---|
| 1 | R | (0,1) | đuôi `(0,0)` rời đi | `[(0,1)]` | 0 |
| 2 | D | (1,1) | đuôi `(0,1)` rời đi | `[(1,1)]` | 0 |
| 3 | R | (1,2) | **ăn mồi 1**, đuôi ở lại | `[(1,1),(1,2)]` | 1 |
| 4 | U | (0,2) | đuôi `(1,1)` rời đi | `[(1,2),(0,2)]` | 1 |
| 5 | L | (0,1) | **ăn mồi 2** | `[(1,2),(0,2),(0,1)]` | 2 |
| 6 | U | (−1,1) | ngoài biên | không đổi | **−1** |

Với ván `2 × 2` (Cách 3): sau ba mồi rắn dài `4` phủ kín bàn; bước `U` về `(0,0)` có `last = 0`, `t = 4`, `t − last = 4 = L` (không `< L`) ⇒ đuôi **đã rời đi**, được phép.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2 (`deque` + `set`)**: mỗi bước `O(1)`, chuẩn mực và dễ giải thích trong phỏng vấn. (Đã đổi tên lớp thành `SnakeGame` đúng skeleton của đề.)

In [16]:
from collections import deque
from typing import List


class SnakeGame:

    def __init__(self, width: int, height: int, food: List[List[int]]):
        self.width = width
        self.height = height
        self.food = [tuple(f) for f in food]       # list [r, c] -> tuple (r, c) để so sánh được
        self.fi = 0                                # chỉ số mồi hiện tại
        self.body = deque([(0, 0)])                # đuôi ở đầu deque, đầu rắn ở cuối
        self.occupied = {(0, 0)}                   # tập ô đang có thân rắn (đồng bộ với body)

    def move(self, direction: str) -> int:
        dr, dc = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}[direction]
        r, c = self.body[-1]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1                              # đâm tường
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1                           # ăn mồi: đuôi ở lại, rắn dài thêm 1
        else:
            self.occupied.remove(self.body.popleft())   # đuôi rời đi trước khi kiểm tra va chạm
        if (nr, nc) in self.occupied:
            return -1                              # đâm vào thân
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return len(self.body) - 1                  # điểm = số mồi đã ăn = độ dài - 1


# Your SnakeGame object will be instantiated and called as such:
# obj = SnakeGame(width, height, food)
# param_1 = obj.move(direction)

print(run_commands(SnakeGame, ops, args), "(kỳ vọng", expected, ")")

[None, 0, 0, 1, 1, 2, -1] (kỳ vọng [None, 0, 0, 1, 1, 2, -1] )


## 10. Vì sao thuật toán đúng

**Bất biến của `deque`/`set`.** Sau mỗi `move` thành công, `body` chứa **đúng các ô của thân rắn** theo thứ tự từ đuôi tới đầu và `occupied` là tập các ô đó. Chứng minh quy nạp: ban đầu `body = [(0,0)]`. Mỗi bước: nếu không ăn thì rắn **mất đuôi** và **thêm đầu mới** (độ dài giữ nguyên) — đúng bằng `popleft` + `remove` rồi `append` + `add`; nếu ăn thì chỉ **thêm đầu mới** (dài thêm 1) — đúng bằng `append` + `add`, không bỏ đuôi.

**Va chạm đúng.** Đề: trò chơi kết thúc nếu đầu chiếm ô mà **thân đang chiếm sau khi di chuyển**. Sau khi đuôi rời đi (nếu có), `occupied` đúng là tập ô thân (không tính đầu mới), nên `(nr, nc) in occupied` chính là "đầu chạm thân". Vì thế rắn dài `4` đi vào ô đuôi cũ vẫn hợp lệ (đuôi đã rời), còn đi vào ô **không phải đuôi** của thân thì thua. Khi ăn mồi, mồi không nằm trên thân (đề đảm bảo) nên đầu mới không thể trùng thân, kiểm tra cũng không báo sai.

**Thứ tự mồi và điểm.** Mồi `i + 1` chỉ xuất hiện sau khi ăn mồi `i` ⇒ chỉ cần so với `food[fi]`. Mỗi mồi ăn làm độ dài tăng `1` nên điểm `= độ dài − 1 = số mồi đã ăn`.

**Cách 3 đúng.** Thân là `L` vị trí đầu gần nhất (mỗi bước thêm một đầu, bỏ một đuôi hoặc giữ đuôi khi ăn), tức các ô có thời điểm `last` trong `[t − L, t − 1]` (với ô đứng nhiều lần, `last` lưu lần **mới nhất** nên không bỏ sót). Khi không ăn, đuôi (thời điểm `t − L`) rời đi, thân còn `last ∈ [t − L + 1, t − 1]` ⇔ `t − last < L`. Điều kiện này khớp đúng "ô còn bị chiếm sau khi đuôi rời".

## 11. Phân tích Time / Space Complexity

Gọi `L` là độ dài rắn, `T` là số lần gọi `move` (`T <= 10⁴`), `K <= T` là số ô khác nhau từng được đầu đi qua.

| Cách | Time mỗi `move` | Space | Ghi chú |
|---|---|---|---|
| 1. `list` (`pop(0)`, `in`) | `O(L)` | `O(L)` | với `L <= 51` vẫn nhanh |
| 2. `deque` + `set` | `O(1)` | `O(L)` | chuẩn mực |
| 3. `dict` thời điểm | `O(1)` | `O(K)` | không cần hàng đợi |

Khởi tạo `O(F)` để đổi `food` thành tuple (`F <= 50`). Tổng thời gian cho cả ván: `O(T)` với Cách 2/3, `O(T · L)` với Cách 1.

## 12. Bộ test

Ta dựng **oracle mô phỏng độc lập** `SnakeOracle`: lưu thân rắn là **list các ô**; mỗi bước **dựng thân mới** (thêm đầu; bỏ đuôi nếu không ăn), rồi kết luận thua nếu thân mới **ra ngoài biên hoặc có ô lặp** (`len(set(body)) != len(body)`) — đúng nguyên văn luật "đầu chạm thân sau khi di chuyển". Nó **khác** hoàn toàn cách kiểm tra từng ô trong các lời giải.

In [17]:
import random
import time

impls = {
    "list": SnakeGameList,
    "deque_demo": SnakeGameDequeDemo,
    "deque": SnakeGameDeque,
    "time_demo": SnakeGameTimeDemo,
    "time": SnakeGameTime,
    "leetcode": SnakeGame,
}


class SnakeOracle:
    """Oracle: dựng thân mới rồi kiểm tra trùng ô / ngoài biên. wrap=True: màn hình xuyên tường (dùng cho biến thể)."""

    def __init__(self, width, height, food, wrap=False):
        self.width, self.height, self.wrap = width, height, wrap
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.body = [(0, 0)]                    # đuôi -> đầu
        self.over = False

    def move(self, direction):
        if self.over:
            return -1
        dr, dc = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}[direction]
        r, c = self.body[-1][0] + dr, self.body[-1][1] + dc
        if self.wrap:
            r, c = r % self.height, c % self.width
        eat = self.fi < len(self.food) and self.food[self.fi] == (r, c)
        new_body = self.body + [(r, c)]
        if eat:
            self.fi += 1
        else:
            new_body = new_body[1:]             # đuôi rời đi
        out = not (0 <= r < self.height and 0 <= c < self.width)
        if out or len(set(new_body)) != len(new_body):
            self.over = True
            return -1
        self.body = new_body
        return len(self.body) - 1


def check_game(width, height, food, dirs, label):
    ops_g = ["SnakeGame"] + ["move"] * len(dirs)
    args_g = [[width, height, [list(f) for f in food]]] + [[d] for d in dirs]
    ref = run_commands(SnakeOracle, ops_g, args_g)
    for key, cls in impls.items():
        got = run_commands(cls, ops_g, args_g)
        assert got == ref, (label, key, width, height, food, dirs, got, ref)
    return ref[1:]                  # bỏ phần tử None của hàm khởi tạo


for key, cls in impls.items():
    assert run_commands(cls, ops, args) == expected, key
assert run_commands(SnakeOracle, ops, args) == expected
print("PASS: ví dụ đề cho tất cả", len(impls), "cách và cho oracle")

PASS: ví dụ đề cho tất cả 6 cách và cho oracle


### 12.1. Các trường hợp biên

Đâm 4 tường, **đuôi rời đi** (rắn dài `4` đi vào ô đuôi — hợp lệ), **tự đâm vào thân**, màn `1 × 1`, và ăn mồi liên tiếp. Các ván `2 × 2` là **tự dựng** (đối chiếu bằng oracle).

In [18]:
# đâm từng tường (rắn xuất phát (0,0) trong màn 3x3)
assert check_game(3, 3, [[2, 2]], "U", "đâm tường trên") == [-1]
assert check_game(3, 3, [[2, 2]], "L", "đâm tường trái") == [-1]
assert check_game(3, 3, [[2, 2]], "RRR", "đâm tường phải") == [0, 0, -1]
assert check_game(3, 3, [[2, 2]], "DDD", "đâm tường dưới") == [0, 0, -1]

# màn 1x1: bước nào cũng ra ngoài
for d in "UDLR":
    assert check_game(1, 1, [[0, 0]], d, "1x1") == [-1]

# ĐUÔI RỜI ĐI: bàn 2x2, rắn dài 4 đi vào ô đuôi (0,0) => hợp lệ; tiếp tục đi vào ô đuôi mới (0,1)
assert check_game(2, 2, [[0, 1], [1, 1], [1, 0]], "RDLURDLU", "đuôi rời đi") == [1, 2, 3, 3, 3, 3, 3, 3]

# TỰ ĐÂM: 3x3, ăn 3 mồi (dài 4) rồi đi vào thân
res = check_game(3, 3, [[0, 1], [0, 2], [1, 2]], "RRDLR", "tự đâm")
assert res == [1, 2, 3, 3, -1]

# ăn mồi liên tiếp trên một hàng
assert check_game(5, 1, [[0, 1], [0, 2], [0, 3], [0, 4]], "RRRR", "liên tiếp") == [1, 2, 3, 4]
# mồi ở xa, đi vòng không ăn
assert check_game(4, 4, [[3, 3]], "RDLU", "đi vòng không ăn") == [0, 0, 0, 0]
print("PASS: các trường hợp biên")

PASS: các trường hợp biên


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi nghe hợp lý: (a) kiểm tra va chạm **trước** khi bỏ đuôi (cấm đi vào ô đuôi); (b) **không xóa đuôi khỏi `set`** khi nó rời đi.

In [19]:
class WrongCheckBeforeTail:
    def __init__(self, width, height, food):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction):
        dr, dc = DIRS[direction]
        nr, nc = self.body[-1][0] + dr, self.body[-1][1] + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1
        if (nr, nc) in self.occupied:                  # SAI: kiểm tra TRƯỚC khi đuôi rời đi
            return -1
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1
        else:
            self.occupied.remove(self.body.popleft())
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return len(self.body) - 1


class WrongStaleSet:
    def __init__(self, width, height, food):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction):
        dr, dc = DIRS[direction]
        nr, nc = self.body[-1][0] + dr, self.body[-1][1] + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1
        else:
            self.body.popleft()                        # SAI: quên occupied.remove(tail)
        if (nr, nc) in self.occupied:
            return -1
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return len(self.body) - 1


loop_ops = ["SnakeGame"] + ["move"] * 5
loop_args = [[2, 2, [[0, 1], [1, 1], [1, 0]]], ["R"], ["D"], ["L"], ["U"], ["R"]]
print("đúng               :", run_commands(SnakeOracle, loop_ops, loop_args))
print("kiểm tra trước đuôi:", run_commands(WrongCheckBeforeTail, loop_ops, loop_args))
back_ops = ["SnakeGame", "move", "move"]
back_args = [[4, 1, [[0, 3]]], ["R"], ["L"]]
print("đúng               :", run_commands(SnakeOracle, back_ops, back_args))
print("set không xóa đuôi :", run_commands(WrongStaleSet, back_ops, back_args))
assert run_commands(WrongCheckBeforeTail, loop_ops, loop_args) != run_commands(SnakeOracle, loop_ops, loop_args)
assert run_commands(WrongStaleSet, back_ops, back_args) != run_commands(SnakeOracle, back_ops, back_args)

đúng               : [None, 1, 2, 3, 3, 3]
kiểm tra trước đuôi: [None, 1, 2, 3, -1, -1]
đúng               : [None, 0, 0]
set không xóa đuôi : [None, 0, -1]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

Sinh **ván hợp lệ theo đúng luật đề**: chọn `width, height` (tổng số ô `>= 2`), mồi xuất hiện **lần lượt**: mồi đầu tiên và mỗi mồi sau khi mồi trước bị ăn đều được chọn ngẫu nhiên trong các **ô trống** (không thuộc thân) tại thời điểm xuất hiện. Hướng đi được chọn ngẫu nhiên, nhưng với xác suất `60%` ta đi **về phía mồi** để rắn ăn được nhiều mồi, đa số lần tránh đâm tường (để còn thử đâm thân), và dừng khi thua hoặc đủ số bước. (Sau khi `move` trả `-1` thì trò chơi kết thúc, không gọi `move` nữa.) Mồi tối đa `50`.

In [20]:
def random_game(rng):
    while True:
        width, height = rng.randint(1, 7), rng.randint(1, 7)
        if width * height >= 2:
            break
    oracle = SnakeOracle(width, height, [])           # dùng làm bộ sinh trạng thái
    food = []

    def spawn():
        free = [(r, c) for r in range(height) for c in range(width) if (r, c) not in oracle.body]
        if not free or len(food) >= 50:
            return None
        cell = rng.choice(free)
        food.append(cell)
        oracle.food.append(cell)                       # oracle biết mồi mới ngay khi nó xuất hiện
        return cell

    current = spawn()
    dirs = []
    for _ in range(rng.randint(1, 80)):
        if current is not None and rng.random() < 0.6:
            hr, hc = oracle.body[-1]
            options = []
            if current[0] < hr: options.append("U")
            if current[0] > hr: options.append("D")
            if current[1] < hc: options.append("L")
            if current[1] > hc: options.append("R")
            d = rng.choice(options) if options else rng.choice("UDLR")
        else:
            d = rng.choice("UDLR")
        if rng.random() < 0.9:                          # đa số thời gian tránh đâm tường để còn thử đâm thân
            hr, hc = oracle.body[-1]
            legal = [x for x in "UDLR" if 0 <= hr + DIRS[x][0] < height and 0 <= hc + DIRS[x][1] < width]
            if d not in legal and legal:
                d = rng.choice(legal)
        dirs.append(d)
        fi_before = oracle.fi
        score = oracle.move(d)
        if score == -1:
            break
        if oracle.fi > fi_before:                      # vừa ăn mồi: mồi kế tiếp xuất hiện
            current = spawn()
    return width, height, [list(f) for f in food], dirs


rng = random.Random(353)
games = lost = 0
total_moves = 0
max_score = 0
for _ in range(3000):
    width, height, food, dirs = random_game(rng)
    assert len(food) <= 50
    if not food:
        continue                                      # đề đảm bảo food.length >= 1
    ref = check_game(width, height, food, dirs, "ngẫu nhiên")
    games += 1
    total_moves += len(dirs)
    lost += ref[-1] == -1
    max_score = max(max_score, max(x for x in ref if x is not None))
print(f"PASS: {games} ván ngẫu nhiên, {total_moves} nước đi, {lost} ván thua, điểm cao nhất {max_score}")

PASS: 3000 ván ngẫu nhiên, 33837 nước đi, 2582 ván thua, điểm cao nhất 9


### 12.4. Kiểm tra hiệu năng (có một phép thử **ngoài** ràng buộc đề để thấy sự khác nhau)

(a) Ở giới hạn đề: `10⁴ × 10⁴`, `50` mồi, `10⁴` bước (đi dọc hàng đầu rồi xuống). (b) **Ngoài ràng buộc**: bàn `100 × 100` đi theo đường **rắn bò** (boustrophedon), `3000` mồi ngay trước đầu rắn, rồi `3000` bước nữa (rắn dài `3001`): ở đây `O(L)` của Cách 1 lộ rõ, trong khi Cách 2/3 vẫn `O(1)`.

In [21]:
# (a) giới hạn của đề
food_a = [[0, i * 100] for i in range(1, 51)]                  # 50 mồi trên hàng 0
dirs_a = "R" * 5000 + "D" * 4000 + "L" * 999                   # 9999 bước, không đâm gì
ops_a = ["SnakeGame"] + ["move"] * len(dirs_a)
args_a = [[10000, 10000, food_a]] + [[d] for d in dirs_a]
ref_a = None
for key in ["list", "deque", "time", "leetcode"]:
    t0 = time.perf_counter()
    out = run_commands(impls[key], ops_a, args_a)
    dt = time.perf_counter() - t0
    ref_a = ref_a or out
    assert out == ref_a and out[-1] == 50, key
    print(f"(a) {key:9s} {len(dirs_a)} bước: {dt * 1000:7.1f} ms, điểm cuối {out[-1]}")

# (b) ngoài ràng buộc: đường rắn bò trên bàn 100 x 100
W = H = 100
path = []
for r in range(H):
    cols = range(W) if r % 2 == 0 else range(W - 1, -1, -1)
    path += [(r, c) for c in cols]
K = 3000
food_b = [list(cell) for cell in path[1:K + 1]]                  # mồi ngay phía trước đầu rắn
steps = []
for (r1, c1), (r2, c2) in zip(path, path[1:K + K + 1]):
    steps.append("D" if r2 > r1 else ("R" if c2 > c1 else "L"))
ops_b = ["SnakeGame"] + ["move"] * len(steps)
args_b = [[W, H, food_b]] + [[d] for d in steps]
expected_b = [None] + list(range(1, K + 1)) + [K] * (len(steps) - K)
for key in ["list", "deque", "time"]:
    t0 = time.perf_counter()
    out = run_commands(impls[key], ops_b, args_b)
    dt = time.perf_counter() - t0
    assert out == expected_b, key
    print(f"(b) {key:9s} {len(steps)} bước, L tới {K + 1}: {dt * 1000:8.1f} ms")

(a) list      9999 bước:     7.6 ms, điểm cuối 50
(a) deque     9999 bước:     5.5 ms, điểm cuối 50
(a) time      9999 bước:     6.2 ms, điểm cuối 50
(a) leetcode  9999 bước:     5.7 ms, điểm cuối 50
(b) list      6000 bước, L tới 3001:    139.9 ms
(b) deque     6000 bước, L tới 3001:      4.5 ms
(b) time      6000 bước, L tới 3001:      3.3 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Kiểm tra va chạm **trước** khi bỏ đuôi

In [22]:
sai = run_commands(WrongCheckBeforeTail, loop_ops, loop_args)
dung = run_commands(SnakeGame, loop_ops, loop_args)
print("sai :", sai)
print("đúng:", dung)
assert sai != dung

sai : [None, 1, 2, 3, -1, -1]
đúng: [None, 1, 2, 3, 3, 3]


**Vì sao sai:** rắn dài `4` trong bàn `2 × 2` đi vào ô đuôi `(0,0)`: đuôi **sẽ rời đi** ngay trong bước này nên ô đó trống. Kiểm tra trước khi bỏ đuôi báo nhầm `-1`. **Cách sửa:** (nếu không ăn mồi) `popleft` + `remove` đuôi **trước**, rồi mới kiểm tra `in occupied`.

### Lỗi 2: Bỏ đuôi khỏi `deque` nhưng quên bỏ khỏi `set`

In [23]:
sai = run_commands(WrongStaleSet, back_ops, back_args)
dung = run_commands(SnakeGame, back_ops, back_args)
print("sai :", sai)
print("đúng:", dung)
assert sai != dung

sai : [None, 0, -1]
đúng: [None, 0, 0]


**Vì sao sai:** `occupied` giữ ô `(0,0)` dù thân không còn ở đó nên quay về `(0,0)` bị báo đâm thân. Hai cấu trúc `deque` và `set` **phải luôn đồng bộ**: mỗi lần thêm/bớt ở `body` thì thêm/bớt ở `occupied`. **Cách sửa:** `self.occupied.remove(self.body.popleft())`.

### Lỗi 3: Nhầm `width` và `height` khi kiểm tra biên

In [24]:
class WrongBounds:
    def __init__(self, width, height, food):
        self.width, self.height = width, height
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction):
        dr, dc = DIRS[direction]
        nr, nc = self.body[-1][0] + dr, self.body[-1][1] + dc
        if not (0 <= nr < self.width and 0 <= nc < self.height):          # SAI: hàng phải so với height, cột với width
            return -1
        self.occupied.remove(self.body.popleft())
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return 0


b_ops = ["SnakeGame", "move", "move"]
b_args = [[3, 2, [[1, 1]]], ["D"], ["D"]]              # màn 3 cột x 2 hàng: D hai lần => hàng 2 ngoài biên
print("sai :", run_commands(WrongBounds, b_ops, b_args))
print("đúng:", run_commands(SnakeGame, b_ops, b_args))
assert run_commands(SnakeGame, b_ops, b_args) == [None, 0, -1]

sai : [None, 0, 0]
đúng: [None, 0, -1]


**Vì sao sai:** `food[i] = (hàng, cột)`, hàng chạy theo **`height`**, cột chạy theo **`width`**, nhưng hàm khởi tạo nhận `(width, height, food)` (**`width` trước**). Đổi chỗ khiến biên dưới cho phép tới hàng `width − 1`. **Cách sửa:** `0 <= nr < self.height and 0 <= nc < self.width`.

### Lỗi 4: So sánh mồi (list) với tọa độ (tuple)

In [25]:
class WrongListVsTuple:
    def __init__(self, width, height, food):
        self.width, self.height, self.food, self.fi = width, height, food, 0      # SAI: food vẫn là list các list
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction):
        dr, dc = DIRS[direction]
        nr, nc = self.body[-1][0] + dr, self.body[-1][1] + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):       # [1, 2] == (1, 2) là False!
            self.fi += 1
        else:
            self.occupied.remove(self.body.popleft())
        if (nr, nc) in self.occupied:
            return -1
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return len(self.body) - 1


print("sai :", run_commands(WrongListVsTuple, ops, args))
print("đúng:", run_commands(SnakeGame, ops, args))
assert run_commands(WrongListVsTuple, ops, args) != expected

sai : [None, 0, 0, 0, 0, 0, -1]
đúng: [None, 0, 0, 1, 1, 2, -1]


**Vì sao sai:** `food[fi]` là **list** `[1, 2]` còn tọa độ rắn là **tuple** `(1, 2)`; `[1, 2] == (1, 2)` là `False`, nên rắn **không bao giờ ăn được mồi** và điểm luôn `0`. **Cách sửa:** đổi mồi sang tuple một lần ở `__init__`: `self.food = [tuple(f) for f in food]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài **thiết kế mô phỏng** có "thân là hàng đợi" (rắn, hàng xếp, cửa sổ trượt...) ⇒ **`deque` + `set`** để `O(1)`.
- **Hai cấu trúc phải đồng bộ**: mỗi thay đổi thân đều cập nhật cả hai (viết gọn: `self.occupied.remove(self.body.popleft())`).
- **Thứ tự**: *di chuyển đuôi → kiểm tra va chạm → thêm đầu*. Khi ăn mồi, **không** di chuyển đuôi.
- Tọa độ lưu bằng **tuple**; đổi đầu vào dạng list ngay trong `__init__`.
- Quy ước `(hàng, cột)` ↔ `(height, width)`; `'U'` làm hàng **giảm**.
- Điểm = độ dài − 1, nên không cần biến điểm riêng.
- Nếu đề có thêm câu hỏi "làm sao `O(1)` mà không dùng hàng đợi", nhớ **thân rắn = `L` vị trí gần nhất của đầu** và ghi **thời điểm** đến mỗi ô (Cách 3).
- Luôn dựng **oracle mô phỏng ngây thơ** (thân là list, kiểm tra trùng) để so sánh với ván ngẫu nhiên.

## 15. Biến thể của bài

1. **Màn hình xuyên tường** (đi ra mép phải thì xuất hiện ở mép trái): chỉ còn thua khi đâm thân (bài tập 5a).
2. **Hiển thị bàn chơi** dạng chuỗi ký tự (bài tập 5b).
3. **Mồi xuất hiện ngẫu nhiên / không giới hạn**: chuyển `food` thành bộ sinh mồi, vẫn kiểm tra không rơi vào thân.
4. **Hai con rắn / nhiều người chơi:** mỗi con có `deque` riêng, `occupied` chung.
5. **Design Phone Directory (379)**, **Design Hit Counter (362)**, **Moving Average from Data Stream (346)**: bài thiết kế dùng `deque` làm cửa sổ.

In [26]:
class SnakeWrap:
    """Biến thể: màn hình xuyên tường; chỉ thua khi đầu chạm thân."""

    def __init__(self, width, height, food):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction):
        dr, dc = DIRS[direction]
        r, c = self.body[-1]
        nr, nc = (r + dr) % self.height, (c + dc) % self.width        # xuyên tường
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1
        else:
            self.occupied.remove(self.body.popleft())
        if (nr, nc) in self.occupied:
            return -1
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return len(self.body) - 1


def render_board(game):
    """Trả về chuỗi vẽ bàn chơi: S = đầu, s = thân, . = trống."""
    grid = [["." for _ in range(game.width)] for _ in range(game.height)]
    for r, c in game.body:
        grid[r][c] = "s"
    r, c = game.body[-1]
    grid[r][c] = "S"
    return "\n".join(" ".join(row) for row in grid)


g = SnakeWrap(3, 2, [[0, 2]])
print([g.move(d) for d in "LRR"])             # L: (0,0)->(0,2) xuyên tường, ăn mồi => 1; hai bước R sau đó: (0,0), (0,1) => 1, 1
print(render_board(g))
game = SnakeGame(3, 2, [[1, 2], [0, 1]])
for d in "RDR":
    game.move(d)
print(render_board(game))

[1, 1, 1]
s S .
. . .
. . .
. s S


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time mỗi `move` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `list` | thân là list, `pop(0)` + `in` | `O(L)` | `O(L)` | bám sát đề, `L` nhỏ (`<= 51` ở đề này) |
| 2. `deque` + `set` | hàng đợi + tập ô chiếm | `O(1)` | `O(L)` | **nên nộp**, kinh điển |
| 3. `dict` thời điểm | thân = `L` vị trí gần nhất của đầu | `O(1)` | `O(K)` | khi muốn không dùng hàng đợi, hoặc cần lịch sử đường đi |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[3, 4] == (3, 4)`, `(3, 4) in {(3, 4)}`, `deque([5, 6, 7]).popleft()`, `DIRS["U"]`, và `run_commands(SnakeGame, ["S", "move"], [[2, 2, [[0, 1]]], ["R"]])`.

**Bài 2 (tính tay).** Bàn `width = 3`, `height = 3`, `food = [[0,1],[0,2],[1,2]]`, các bước `R, R, D, L, R`. Lập bảng `body` (đuôi → đầu) và điểm sau từng bước; chỉ ra bước nào thua và vì sao.

**Bài 3 (điền TODO).** Hoàn thành `SnakeGameTodo` theo Cách 2 (`deque` + `set`).

In [27]:
class SnakeGameTodo:
    def __init__(self, width, height, food):
        # TODO 1: lưu width, height; đổi food thành list tuple; fi = 0
        # TODO 2: body = deque([(0, 0)]); occupied = {(0, 0)}
        pass

    def move(self, direction):
        # TODO 3: tính đầu mới (nr, nc) từ body[-1] và hướng; ngoài biên -> trả -1
        # TODO 4: nếu trùng food[fi]: fi += 1; ngược lại popleft đuôi và remove khỏi occupied
        # TODO 5: nếu (nr, nc) in occupied -> trả -1
        # TODO 6: append, add; trả len(body) - 1
        pass

# KHÔNG tạo đối tượng / gọi move ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết lớp `SnakeGameTime2` theo Cách 3 (ghi `last[ô] = t`, độ dài `L`; va chạm khi `t - last[ô] < L`) **mà không nhìn lại code**.

**Bài 5 (biến thể / mini challenge).** (a) `SnakeWrapAns` (màn xuyên tường): dùng `%` cho hàng/cột, chỉ thua khi đâm thân; kiểm tra với `SnakeOracle(..., wrap=True)`. (b) `board_string(width, height, cells)`: trả chuỗi vẽ bàn chơi cho danh sách ô thân (đuôi → đầu): đầu `S`, thân `s`, còn lại `.`.

In [28]:
class SnakeWrapTodo:
    def __init__(self, width, height, food):
        # TODO: giống Cách 2 nhưng tọa độ mới lấy % height, % width (không bao giờ ra biên)
        pass

    def move(self, direction):
        pass


def board_string_todo(width, height, cells):
    # TODO: tạo lưới height x width toàn '.', đặt 's' cho các ô thân, 'S' cho ô cuối cùng của cells
    pass

# KHÔNG gọi các hàm / lớp TODO ở đây.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[3, 4] == (3, 4)` là `False` (khác kiểu); `(3, 4) in {(3, 4)}` là `True`; `popleft` lấy `5`; `DIRS["U"] == (-1, 0)`. Bài 3: nhớ `food` đổi thành tuple và `deque`/`set` luôn đồng bộ.
- **Mức 2 (vừa):** Bài 2: ba mồi đầu tiên cho điểm `1, 2, 3`, thân dài `4` kín `(0,0)`, `(0,1)`, `(0,2)`, `(1,2)`; bước `L` đi tới `(1,1)` (đuôi `(0,0)` rời đi); bước `R` đi lại `(1,2)`. Bài 5a: `nr = (r + dr) % height`, `nc = (c + dc) % width`; `%` của số âm trong Python vẫn không âm.
- **Mức 3 (sâu):** Bài 2 kết quả `[1, 2, 3, 3, -1]`: ở bước 5, đầu mới `(1,2)` là ô **thân** (không phải đuôi: đuôi lúc đó là `(0,1)`), nên thua. Bài 4: `t` tăng mỗi lần đi hợp lệ; khi ăn mồi `L += 1` và **không** kiểm tra va chạm; va chạm chỉ xét khi không ăn.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert ([3, 4] == (3, 4)) is False
assert (3, 4) in {(3, 4)}
assert deque([5, 6, 7]).popleft() == 5
assert DIRS["U"] == (-1, 0)
assert run_commands(SnakeGame, ["S", "move"], [[2, 2, [[0, 1]]], ["R"]]) == [None, 1]

# Bài 2
res2 = run_commands(SnakeGame, ["S"] + ["move"] * 5, [[3, 3, [[0, 1], [0, 2], [1, 2]]], ["R"], ["R"], ["D"], ["L"], ["R"]])
assert res2 == [None, 1, 2, 3, 3, -1]
assert run_commands(SnakeOracle, ["S"] + ["move"] * 5, [[3, 3, [[0, 1], [0, 2], [1, 2]]], ["R"], ["R"], ["D"], ["L"], ["R"]]) == res2

# Bài 3
class SnakeGameTodoAns:
    def __init__(self, width, height, food):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction):
        dr, dc = DIRS[direction]
        nr, nc = self.body[-1][0] + dr, self.body[-1][1] + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1
        else:
            self.occupied.remove(self.body.popleft())
        if (nr, nc) in self.occupied:
            return -1
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return len(self.body) - 1

assert run_commands(SnakeGameTodoAns, ops, args) == expected

# Bài 4
class SnakeGameTime2:
    def __init__(self, width, height, food):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi, self.L, self.t = 0, 1, 0
        self.head = (0, 0)
        self.last = {(0, 0): 0}

    def move(self, direction):
        dr, dc = DIRS[direction]
        cell = (self.head[0] + dr, self.head[1] + dc)
        if not (0 <= cell[0] < self.height and 0 <= cell[1] < self.width):
            return -1
        self.t += 1
        if self.fi < len(self.food) and self.food[self.fi] == cell:
            self.fi += 1
            self.L += 1
        elif cell in self.last and self.t - self.last[cell] < self.L:
            return -1
        self.last[cell] = self.t
        self.head = cell
        return self.L - 1

# Bài 5a
class SnakeWrapAns:
    def __init__(self, width, height, food):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction):
        dr, dc = DIRS[direction]
        r, c = self.body[-1]
        nr, nc = (r + dr) % self.height, (c + dc) % self.width
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1
        else:
            self.occupied.remove(self.body.popleft())
        if (nr, nc) in self.occupied:
            return -1
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return len(self.body) - 1

class WrapOracle(SnakeOracle):
    def __init__(self, width, height, food):
        super().__init__(width, height, food, wrap=True)

rng_ans = random.Random(3530)
for _ in range(1500):
    width, height, food, dirs = random_game(rng_ans)
    if not food:
        continue
    ops_g = ["S"] + ["move"] * len(dirs)
    args_g = [[width, height, food]] + [[d] for d in dirs]
    assert run_commands(SnakeGameTodoAns, ops_g, args_g) == run_commands(SnakeOracle, ops_g, args_g)
    assert run_commands(SnakeGameTime2, ops_g, args_g) == run_commands(SnakeOracle, ops_g, args_g)
    # biến thể xuyên tường: so với oracle wrap trên một chuỗi hướng đi ngẫu nhiên khác
    dirs_w = [rng_ans.choice("UDLR") for _ in range(60)]
    ops_w = ["S"] + ["move"] * len(dirs_w)
    args_w = [[width, height, food]] + [[d] for d in dirs_w]
    got_w, ref_w = run_commands(SnakeWrapAns, ops_w, args_w), run_commands(WrapOracle, ops_w, args_w)
    cut = ref_w.index(-1) + 1 if -1 in ref_w else len(ref_w)      # sau khi thua không còn gọi move nữa
    assert got_w[:cut] == ref_w[:cut]

# Bài 5b
def board_string(width, height, cells):
    grid = [["." for _ in range(width)] for _ in range(height)]
    for r, c in cells:
        grid[r][c] = "s"
    r, c = cells[-1]
    grid[r][c] = "S"
    return "\n".join(" ".join(row) for row in grid)

assert board_string(3, 2, [(1, 1), (1, 2)]) == ". . .\n. s S"
assert board_string(2, 2, [(0, 0)]) == "S .\n. ."
print("Tất cả đáp án bài tập đều pass")

Tất cả đáp án bài tập đều pass


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (`deque` + `set`)

```python
class SnakeGame:

    def __init__(self, width: int, height: int, food: List[List[int]]):
        self.width, self.height = width, height
        self.food = [tuple(f) for f in food]
        self.fi = 0
        self.body = deque([(0, 0)])
        self.occupied = {(0, 0)}

    def move(self, direction: str) -> int:
        dr, dc = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}[direction]
        r, c = self.body[-1]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < self.height and 0 <= nc < self.width):
            return -1
        if self.fi < len(self.food) and self.food[self.fi] == (nr, nc):
            self.fi += 1
        else:
            self.occupied.remove(self.body.popleft())
        if (nr, nc) in self.occupied:
            return -1
        self.body.append((nr, nc))
        self.occupied.add((nr, nc))
        return len(self.body) - 1
```

**Checklist**

- [ ] `food[i]` đã đổi sang **tuple** để so sánh với `(nr, nc)`?
- [ ] Biên: `0 <= nr < height` (hàng) và `0 <= nc < width` (cột), `'U'` làm hàng giảm?
- [ ] Không ăn mồi ⇒ `popleft` đuôi **và** `remove` khỏi `set` **trước** khi kiểm tra va chạm?
- [ ] Ăn mồi ⇒ đuôi **ở lại**, tăng chỉ số mồi?
- [ ] `deque` và `set` luôn **đồng bộ**; đầu mới được `append` + `add` cuối cùng?
- [ ] Điểm = `len(body) - 1`; trả `-1` khi thua?
- [ ] Đã thử đuôi rời đi trên bàn `2 × 2` (rắn dài `4` vẫn đi vào ô đuôi) và ví dụ đề?
- [ ] Skeleton đúng `class SnakeGame` / `__init__(self, width: int, height: int, food: List[List[int]])` / `move(self, direction: str) -> int`?